<a href="https://colab.research.google.com/github/Maxs-Cloud/Stem/blob/master/Untitled6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from demucs_infer.pretrained import get_model
from demucs_infer.apply import apply_model
import numpy as np
import soundfile as sf
import librosa
from pathlib import Path
from typing import Dict
import warnings
warnings.filterwarnings('ignore')

class DemucsSeparator:
    """
    Класс для разделения аудио на стемы с помощью demucs-infer.
    """

    def __init__(self, model_name='htdemucs_ft', device=None):
        self.model_name = model_name
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.sample_rate = 44100

        print(f"Загрузка модели {model_name} на {self.device}...")

        self.model = get_model(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.apply_model = apply_model
        print("Модель загружена!")

    def separate(self, audio_path: str, output_dir: str = None) -> Dict[str, np.ndarray]:
        """
        Разделяет аудиофайл на стемы.

        Порядок стемов от demucs-infer: drums, bass, other, vocals
        """
        print(f"Загрузка аудио: {audio_path}")
        audio, sr = librosa.load(audio_path, sr=self.sample_rate, mono=False)

        if audio.ndim == 1:
            audio = audio[np.newaxis, :]
        elif audio.ndim == 2 and audio.shape[0] > 2:
            audio = audio[:2, :]

        audio_tensor = torch.from_numpy(audio).float().to(self.device).unsqueeze(0)

        print("Разделение на стемы...")
        with torch.no_grad():
            sources = self.apply_model(
                self.model,
                audio_tensor,
                device=self.device,
                shifts=1,
                split=True,
                overlap=0.25
            )

        result = {
            'drums': sources[0, 0].cpu().numpy(),
            'bass': sources[0, 1].cpu().numpy(),
            'other': sources[0, 2].cpu().numpy(),
            'vocals': sources[0, 3].cpu().numpy()
        }

        if output_dir:
            output_dir = Path(output_dir)
            output_dir.mkdir(parents=True, exist_ok=True)
            for name, audio_array in result.items():
                output_path = output_dir / f"{name}.wav"
                sf.write(str(output_path), audio_array.T, self.sample_rate)
                print(f"Сохранен стем: {output_path}")

        return result

KeyboardInterrupt: 

In [ ]:
!pip install museval musdb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 963.2/963.2 kB 23.7 MB/s eta 0:00:00


In [3]:
import museval
import numpy as np
import musdb
from pathlib import Path
from typing import Dict
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import tempfile
import soundfile as sf

warnings.filterwarnings('ignore')

class MetricsEvaluator:
    """
    Класс для оценки качества разделения на датасете MUSDB18.
    Принимает готовый объект musdb.DB.
    """

    def __init__(self, db):
        """
        Args:
            db: объект musdb.DB, загруженный с subsets='test' и download=True
        """
        self.db = db
        self.stem_names = ['vocals', 'drums', 'bass', 'other']

    def evaluate_track(self, reference: Dict[str, np.ndarray],
                       estimated: Dict[str, np.ndarray]) -> Dict[str, Dict[str, float]]:
        """
        Оценка метрик для одного трека.

        Args:
            reference: эталонные стемы в формате (samples, channels)
            estimated: предсказанные стемы в формате (channels, samples)
        """
        # Приводим оба к формату (sources, samples, channels)
        ref_list = []
        est_list = []

        for name in self.stem_names:
            # Эталон уже в правильном формате (samples, channels)
            ref = reference[name]

            # Предсказание нужно транспонировать: (channels, samples) → (samples, channels)
            est = estimated[name]
            if est.ndim == 2 and est.shape[0] == 2 and est.shape[1] > 2:
                est = est.T

            ref_list.append(ref)
            est_list.append(est)

        # Собираем матрицы (4, samples, 2)
        ref_matrix = np.stack(ref_list, axis=0)
        est_matrix = np.stack(est_list, axis=0)

        # Вычисляем метрики
        sdr, isr, sir, sar, _ = museval.metrics.bss_eval(
            ref_matrix,
            est_matrix,
            compute_permutation=False
        )

        # Собираем результат: медиана по всем временным окнам
        results = {}
        for i, name in enumerate(self.stem_names):
            results[name] = {
                'SDR': float(np.nanmedian(sdr[i])),
                'SIR': float(np.nanmedian(sir[i])),
                'SAR': float(np.nanmedian(sar[i])),
                'ISR': float(np.nanmedian(isr[i]))
            }
        return results

    def evaluate_dataset(self, separator, num_tracks: int = 10) -> pd.DataFrame:
        """
        Оценка на num_tracks треках из MUSDB18.

        Args:
            separator: экземпляр DemucsSeparator с методом separate()
            num_tracks: количество треков для оценки

        Returns:
            DataFrame с усреднёнными метриками по стемам
        """
        db = self.db
        all_results = []

        for track in tqdm(db[:num_tracks], desc="Оценка треков"):
            # 1. Эталонные стемы из датасета — формат (samples, channels)
            reference = {
                'vocals': track.targets['vocals'].audio,
                'drums': track.targets['drums'].audio,
                'bass': track.targets['bass'].audio,
                'other': track.audio - track.targets['vocals'].audio
                         - track.targets['drums'].audio
                         - track.targets['bass'].audio
            }

            # 2. Сохраняем микс во временный WAV
            mix = track.audio  # (samples, channels)
            with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                sf.write(tmp.name, mix, 44100)
                tmp_path = tmp.name

            # 3. Разделяем — получаем стемы в формате (channels, samples)
            estimated = separator.separate(tmp_path)

            # 4. Сравниваем (транспонирование внутри evaluate_track)
            track_results = self.evaluate_track(reference, estimated)

            # 5. Удаляем временный файл
            Path(tmp_path).unlink()

            # 6. Добавляем метрики в общий список
            for stem_name, metrics in track_results.items():
                metrics['track'] = track.name
                metrics['stem'] = stem_name
                all_results.append(metrics)

        # Защита от пустого списка
        if not all_results:
            print("⚠️ Нет данных для оценки. Проверьте, что датасет загружен правильно.")
            return pd.DataFrame()

        # Усредняем метрики по стемам
        df = pd.DataFrame(all_results)
        summary = df.groupby('stem')[['SDR', 'SIR', 'SAR', 'ISR']].mean().reset_index()

        # Визуализация
        self.plot_metrics(summary)

        return summary

    def plot_metrics(self, df: pd.DataFrame):
        """Визуализация метрик в виде столбчатых диаграмм."""
        metrics = ['SDR', 'SIR', 'SAR', 'ISR']
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        for idx, metric in enumerate(metrics):
            ax = axes[idx // 2, idx % 2]
            bars = ax.bar(df['stem'], df[metric])
            ax.set_title(f'{metric} по стемам (dB)')
            ax.set_ylabel('dB')
            ax.set_ylim(-10, 20)

            # Подписи значений над столбцами
            for bar, value in zip(bars, df[metric]):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                        f'{value:.1f}', ha='center', va='bottom')

        plt.tight_layout()
        plt.savefig('metrics_summary.png', dpi=150)
        print("График сохранён: metrics_summary.png")

        # Таблица в консоли
        print("\n" + "=" * 60)
        print("СРЕДНИЕ МЕТРИКИ ПО СТЕМАМ (dB)")
        print("=" * 60)
        print(df.to_string(index=False))
        print("=" * 60)

In [4]:
!pip install musdb stempeg

import musdb
import os

# Отдельная чистая папка для тестового набора
data_dir = '/content/musdb18_test'

# Загружаем ТОЛЬКО test (50 треков) с принудительным скачиванием
db_test = musdb.DB(root=data_dir, subsets='test', download=True)

print(f"Количество треков в тесте: {len(db_test)}")
# Проверим первый трек
track = db_test[0]
print(f"Пример: {track.name}")  # если всё хорошо, покажет название трека
print("Доступные цели:", list(track.targets.keys()))  # должно быть ['vocals', 'drums', 'bass', 'other']

Количество треков в тесте: 50
Пример: AM Contra - Heart Peripheral
Доступные цели: ['vocals', 'drums', 'bass', 'other', 'accompaniment', 'linear_mixture']


In [1]:
from fastapi import FastAPI, File, UploadFile, HTTPException, Request
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import tempfile
import shutil
from pathlib import Path
from typing import Dict
import zipfile
import uuid
import time
import os
import torch
import numpy as np
import soundfile as sf
import librosa
import warnings
warnings.filterwarnings('ignore')

# ==================== Класс DemucsSeparator ====================
class DemucsSeparator:
    """Класс для разделения аудио через demucs-infer."""

    def __init__(self, model_name='htdemucs_ft', device=None):
        self.model_name = model_name
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
        self.sample_rate = 44100

        print(f"Загрузка модели {model_name} на {self.device}...")
        from demucs_infer.pretrained import get_model
        from demucs_infer.apply import apply_model

        self.model = get_model(model_name)
        self.model.to(self.device)
        self.model.eval()
        self.apply_model = apply_model
        print("Модель загружена!")

    def separate(self, audio_path: str, output_dir: str = None) -> Dict[str, np.ndarray]:
        print(f"Загрузка аудио: {audio_path}")
        audio, sr = librosa.load(audio_path, sr=self.sample_rate, mono=False)

        # Если файл моно — дублируем канал
        if audio.ndim == 1:
           audio = np.stack([audio, audio])  # Делаем псевдо-стерео
        elif audio.ndim == 2 and audio.shape[0] > 2:
            audio = audio[:2, :]

        audio_tensor = torch.from_numpy(audio).float().to(self.device).unsqueeze(0)

        print("Разделение на стемы...")
        with torch.no_grad():
            sources = self.apply_model(
                self.model,
                audio_tensor,
                device=self.device,
                shifts=1,
                split=True,
                overlap=0.25
            )

        result = {
            'drums': sources[0, 0].cpu().numpy(),
            'bass': sources[0, 1].cpu().numpy(),
            'other': sources[0, 2].cpu().numpy(),
            'vocals': sources[0, 3].cpu().numpy()
        }

        if output_dir:
            output_dir = Path(output_dir)
            output_dir.mkdir(parents=True, exist_ok=True)
            for name, audio_array in result.items():
                output_path = output_dir / f"{name}.wav"
                sf.write(str(output_path), audio_array.T, self.sample_rate)
                print(f"Сохранен стем: {output_path}")

        return result

# ==================== FastAPI App ====================
app = FastAPI(
    title="Music Source Separator API",
    description="API для разделения музыки на инструментальные дорожки",
    version="1.0.0"
)

# CORS
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Глобальная переменная модели
separator = None

@app.on_event("startup")
async def load_model():
    """Загрузка модели при старте сервера."""
    global separator
    separator = DemucsSeparator(model_name='htdemucs_ft')

def validate_audio(filename: str) -> bool:
    """Валидация аудиофайла."""
    allowed_extensions = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
    ext = Path(filename).suffix.lower()
    if ext not in allowed_extensions:
        raise HTTPException(
            status_code=400,
            detail=f"Неподдерживаемый формат: {ext}. Поддерживаются: {allowed_extensions}"
        )
    return True

def check_audio_duration(file_path: str, max_duration_min: int = 10) -> bool:
    """Проверка длительности аудио."""
    import librosa
    try:
        duration = librosa.get_duration(filename=file_path)
        if duration > max_duration_min * 60:
            raise HTTPException(
                status_code=400,
                detail=f"Файл слишком длинный: {duration/60:.1f} мин. Максимум: {max_duration_min} мин."
            )
        return True
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Ошибка чтения аудио: {str(e)}")

@app.post("/separate")
async def separate_audio(file: UploadFile = File(...), request:Request=None):
    """Разделение аудиофайла на стемы."""
    validate_audio(file.filename)

    request_id = str(uuid.uuid4())
    temp_dir = Path(tempfile.gettempdir()) / f"music_sep_{request_id}"
    temp_dir.mkdir(parents=True, exist_ok=True)

    try:
        input_path = temp_dir / file.filename
        with open(input_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)

        check_audio_duration(str(input_path))

        start_time = time.time()
        stems = separator.separate(str(input_path))
        processing_time = time.time() - start_time

        output_dir = temp_dir / "stems"
        stems_paths = {}
        for name, audio in stems.items():
            output_path = output_dir / f"{name}.wav"
            output_path.parent.mkdir(exist_ok=True)
            sf.write(str(output_path), audio.T, separator.sample_rate)
            stems_paths[name] = str(output_path)

        zip_path = temp_dir / "stems.zip"
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for stem_name, stem_path in stems_paths.items():
                zipf.write(stem_path, f"{Path(file.filename).stem}_{stem_name}.wav")

        #
    # Получаем базовый URL из запроса
        base_url = str(request.base_url).rstrip("/")

        return JSONResponse({
        'status': 'success',
        'request_id': request_id,
        'processing_time_seconds': round(processing_time, 2),
        'stems': {
            'vocals': f"{base_url}/download/{request_id}/vocals",
            'drums': f"{base_url}/download/{request_id}/drums",
            'bass': f"{base_url}/download/{request_id}/bass",
            'other': f"{base_url}/download/{request_id}/other"
        },
        'zip_archive': f"{base_url}/download/{request_id}/archive",
        'info': {
            'model': separator.model_name,
            'device': separator.device,
            'sample_rate': separator.sample_rate
        }
    })
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Ошибка обработки: {str(e)}")

@app.get("/download/{request_id}/{stem_name}")
async def download_stem(request_id: str, stem_name: str):
    """Скачивание отдельного стема."""
    temp_dir = Path(tempfile.gettempdir()) / f"music_sep_{request_id}"

    if stem_name == "archive":
        file_path = temp_dir / "stems.zip"
        media_type = "application/zip"
        filename = "stems.zip"
    else:
        file_path = temp_dir / "stems" / f"{stem_name}.wav"
        media_type = "audio/wav"
        filename = f"{stem_name}.wav"

    if not file_path.exists():
        raise HTTPException(status_code=404, detail="Результат не найден")

    return FileResponse(path=str(file_path), media_type=media_type, filename=filename)

@app.get("/health")
async def health_check():
    """Проверка работоспособности."""
    if separator is None:
        return {'status': 'model not loaded'}
    return {
        'status': 'healthy',
        'model': separator.model_name,
        'device': separator.device,
        'gpu_available': torch.cuda.is_available()
    }

@app.get("/")
async def root():
    """Корневой эндпоинт."""
    return {
        'message': 'Music Source Separator API',
        'docs': '/docs',
        'endpoints': {
            'GET /health': 'Проверка состояния',
            'POST /separate': 'Разделение аудио (multipart/form-data)',
            'GET /download/{request_id}/{stem_name}': 'Скачивание результата'
        }
    }

In [7]:
import nest_asyncio
from pyngrok import ngrok
import uvicorn
from fastapi import FastAPI
import asyncio
import threading

ngrok.kill()
nest_asyncio.apply()

# Настраиваем ngrok с вашим токеном (замените на реальный!)
NGROK_AUTH_TOKEN = "3DLYA8pj9VbxVT8lWE76d8SHys4_3aTFwJ44aiX4VV2HDm2LD"  # <-- замените!
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# Создаём туннель
public_url = ngrok.connect(8000)
print(f"\n{'='*60}")
print(f"API доступен по адресу: {public_url}")
print(f"Swagger docs: {public_url}/docs")
print(f"{'='*60}\n")

# Запускаем uvicorn как асинхронный сервер внутри текущего event loop
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Запускаем сервер в фоновом потоке (не блокирует выполнение ячейки)
async def start_server():
    await server.serve()

def run_in_thread():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start_server())

thread = threading.Thread(target=run_in_thread, daemon=True)
thread.start()

print("Сервер запущен в фоновом режиме!")


API доступен по адресу: NgrokTunnel: "https://reapply-tadpole-annually.ngrok-free.dev" -> "http://localhost:8000"
Swagger docs: NgrokTunnel: "https://reapply-tadpole-annually.ngrok-free.dev" -> "http://localhost:8000"/docs

Сервер запущен в фоновом режиме!


In [14]:
#!/usr/bin/env python3
"""
Скрипт для оценки метрик на MUSDB18.
"""
import argparse
#from model import DemucsSeparator
#from metrics import MetricsEvaluator

def main():
    parser = argparse.ArgumentParser(description='Оценка качества разделения аудио')
    parser.add_argument('--musdb-path', type=str, required=True,
                       help='Путь к датасету MUSDB18')
    parser.add_argument('--num-tracks', type=int, default=10,
                       help='Количество треков для оценки (по умолчанию: 10)')
    parser.add_argument('--model', type=str, default='htdemucs',
                       help='Модель Demucs (по умолчанию: htdemucs)')
    parser.add_argument('--device', type=str, default=None,
                       help='Устройство: cuda или cpu (по умолчанию: авто)')

    args = parser.parse_args()

    # Инициализируем разделитель
    print("Инициализация модели...")
    separator = DemucsSeparator(
        model_name=args.model,
        device=args.device
    )

    # Создаем эвалюатор
    evaluator = MetricsEvaluator(musdb_path=args.musdb_path)

    # Запускаем оценку
    print(f"\nОценка на {args.num_tracks} треках из MUSDB18...")
    results = evaluator.evaluate_dataset(separator, num_tracks=args.num_tracks)

    print("\nОценка завершена!")
    print(f"Результаты сохранены в metrics_summary.png")

if __name__ == "__main__":
  #  main()
   separator = DemucsSeparator(model_name='htdemucs', device=device)# Создаём эвалюатор, передавая уже готовый db_test
   evaluator = MetricsEvaluator(db_test)

# Оценка на 10 треках
   results = evaluator.evaluate_dataset(separator, num_tracks=10)
   print(results)

NameError: name 'device' is not defined

In [5]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
import asyncio
import threading
import time

nest_asyncio.apply()


# Настройка uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

# Запуск в фоновом потоке
async def start():
    await server.serve()

def run():
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(start())

thread = threading.Thread(target=run, daemon=True)
thread.start()

# Ждём запуск
time.sleep(3)

# Проверяем
import requests
try:
    response = requests.get("http://localhost:8000/", timeout=5)
    print(f"Сервер работает! Статус: {response.status_code}")
    print(f"Ответ: {response.json()}")
except Exception as e:
    print(f"Сервер не отвечает: {e}")

INFO:     Started server process [135575]
INFO:     Waiting for application startup.


Загрузка модели htdemucs_ft на cpu...
INFO:     127.0.0.1:54750 - "GET / HTTP/1.1" 200 OK
Сервер работает! Статус: 200
Ответ: {'message': 'Music Source Separator API', 'docs': '/docs', 'endpoints': {'GET /health': 'Проверка состояния', 'POST /separate': 'Разделение аудио (multipart/form-data)', 'GET /download/{request_id}/{stem_name}': 'Скачивание результата'}}


In [3]:
from google.colab.output import eval_js

colab_url = eval_js("google.colab.kernel.proxyPort(8000)")
print(f"API: {colab_url}")
print(f"Swagger: {colab_url}docs")

API: https://8000-m-s-kkb-euw4b0-54pjj53oue3x-b.europe-west4-0.prod.colab.dev
Swagger: https://8000-m-s-kkb-euw4b0-54pjj53oue3x-b.europe-west4-0.prod.colab.devdocs


In [4]:
from google.colab.output import eval_js
print(eval_js("google.colab.kernel.proxyPort(8000)"))

https://8000-m-s-kkb-euw4b0-54pjj53oue3x-b.europe-west4-0.prod.colab.dev


In [ ]:
from fastapi import FastAPI, File, UploadFile, HTTPException, Request
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
import tempfile
import shutil
from pathlib import Path
from typing import Dict
import zipfile
import uuid
import time
import torch
import soundfile as sf
import librosa
import warnings
warnings.filterwarnings('ignore')

# Импорт вашего класса из отдельного файла
from model import DemucsSeparator

# ==================== FastAPI App ====================
app = FastAPI(
    title="Music Source Separator API",
    description="API для разделения музыки на инструментальные дорожки",
    version="1.0.0"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

separator = None

@app.on_event("startup")
async def load_model():
    global separator
    separator = DemucsSeparator(model_name='htdemucs_ft')

def validate_audio(filename: str) -> bool:
    allowed_extensions = {'.wav', '.mp3', '.flac', '.ogg', '.m4a'}
    ext = Path(filename).suffix.lower()
    if ext not in allowed_extensions:
        raise HTTPException(
            status_code=400,
            detail=f"Неподдерживаемый формат: {ext}"
        )
    return True

def check_audio_duration(file_path: str, max_duration_min: int = 10) -> bool:
    try:
        duration = librosa.get_duration(filename=file_path)
        if duration > max_duration_min * 60:
            raise HTTPException(
                status_code=400,
                detail=f"Файл слишком длинный: {duration/60:.1f} мин"
            )
        return True
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Ошибка: {str(e)}")

@app.post("/separate")
async def separate_audio(file: UploadFile = File(...), request: Request = None):
    validate_audio(file.filename)

    request_id = str(uuid.uuid4())
    temp_dir = Path(tempfile.gettempdir()) / f"music_sep_{request_id}"
    temp_dir.mkdir(parents=True, exist_ok=True)

    try:
        input_path = temp_dir / file.filename
        with open(input_path, "wb") as buffer:
            shutil.copyfileobj(file.file, buffer)

        check_audio_duration(str(input_path))

        start_time = time.time()
        stems = separator.separate(str(input_path))
        processing_time = time.time() - start_time

        output_dir = temp_dir / "stems"
        stems_paths = {}
        for name, audio in stems.items():
            output_path = output_dir / f"{name}.wav"
            output_path.parent.mkdir(exist_ok=True)
            sf.write(str(output_path), audio.T, separator.sample_rate)
            stems_paths[name] = str(output_path)

        zip_path = temp_dir / "stems.zip"
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for stem_name, stem_path in stems_paths.items():
                zipf.write(stem_path, f"{Path(file.filename).stem}_{stem_name}.wav")

        base_url = str(request.base_url).rstrip("/") if request else "http://localhost:8000"

        return JSONResponse({
            'status': 'success',
            'request_id': request_id,
            'processing_time_seconds': round(processing_time, 2),
            'stems': {
                'vocals': f"{base_url}/download/{request_id}/vocals",
                'drums': f"{base_url}/download/{request_id}/drums",
                'bass': f"{base_url}/download/{request_id}/bass",
                'other': f"{base_url}/download/{request_id}/other"
            },
            'zip_archive': f"{base_url}/download/{request_id}/archive",
            'info': {
                'model': separator.model_name,
                'device': separator.device,
                'sample_rate': separator.sample_rate
            }
        })
    except HTTPException:
        raise
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Ошибка обработки: {str(e)}")

@app.get("/download/{request_id}/{stem_name}")
async def download_stem(request_id: str, stem_name: str):
    temp_dir = Path(tempfile.gettempdir()) / f"music_sep_{request_id}"

    if stem_name == "archive":
        file_path = temp_dir / "stems.zip"
        media_type = "application/zip"
        filename = "stems.zip"
    else:
        file_path = temp_dir / "stems" / f"{stem_name}.wav"
        media_type = "audio/wav"
        filename = f"{stem_name}.wav"

    if not file_path.exists():
        raise HTTPException(status_code=404, detail="Результат не найден")

    return FileResponse(path=str(file_path), media_type=media_type, filename=filename)

@app.get("/health")
async def health_check():
    if separator is None:
        return {'status': 'model not loaded'}
    return {
        'status': 'healthy',
        'model': separator.model_name,
        'device': separator.device,
        'gpu_available': torch.cuda.is_available()
    }

@app.get("/")
async def root():
    return {
        'message': 'Music Source Separator API',
        'docs': '/docs',
        'endpoints': {
            'GET /': 'Документация',
            'GET /health': 'Проверка состояния',
            'POST /separate': 'Разделение аудио',
            'GET /download/{{id}}/{{stem}}': 'Скачивание результата'
        }
    }